In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics #yolo 다운

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.4 MB/s eta 0:00:00


In [3]:
# zip 폴더 해제

%mkdir dataset
!cp /content/drive/MyDrive/rokey_face/my_data.zip /content/
!unzip /content/my_data.zip -d dataset/

Archive:  /content/my_data.zip
   creating: dataset/my_data/
   creating: dataset/my_data/test/
   creating: dataset/my_data/test/labels/
  inflating: dataset/my_data/test/labels/webcamimage_mix94.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix48.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix36.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix9.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix27.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix43.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix8.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix16.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix35.txt  
  inflating: dataset/my_data/test/labels/webcamimage_mix20.txt  
   creating: dataset/my_data/test/images/
  inflating: dataset/my_data/test/images/webcamimage_mix94.jpg  
  inflating: dataset/my_data/test/images/webcamimage_mix9.jpg  
  inflating: dataset/my_data/test/images/we

In [4]:
# yaml 파일 만들기

import yaml

data = {
        # 'train' : '/content/dataset/my_data/train/images',
        # 'test' : '/content/dataset/my_data/test/images',
        # 'val' : '/content/dataset/my_data/valid/images',
        'train' : '/content/dataset/my_data/train/images',
        'test' : '/content/dataset/my_data/test/images',
        'val' : '/content/dataset/my_data/valid/images',

        'nc': 2,
        'names': ['Car', 'Dummy']
        # 'names': ['Truck','Dummy']
        # 'names': ['Normal','Error']
        # 'names' : ['NG', 'OK']
}

# with open('/content/dataset/my_data/custom_data.yaml', 'w') as f:
with open('/content/dataset/my_data/custom_data.yaml', 'w') as f:
  yaml.dump(data, f)

## 파라미터

In [ ]:
# 파라미터

YOLO_NUM = "8"
EPOCHS = 100
PATIENCE = 25
BATCH_SIZE = 32
IMG_SIZE = 1024
FOLDER_ID = "2" # 첫번째는 ''아무것도 입력하지 않는다. 이후엔 2부터 시작해 숫차를 채워 넣는다.


PROJECT_NAME = f"web_y{YOLO_NUM}_b{BATCH_SIZE}_p{PATIENCE}"
DRIVE_PATH = "/content/drive/MyDrive/rokey_face/result" # 이 부분 수정 저장할 공간


from ultralytics import YOLO

if YOLO_NUM == "26":
  model_name = f"yolo26n.pt"
else:
  model_name = f"yolov{YOLO_NUM}n.pt"

model = YOLO(model_name)

In [ ]:
''' 원클릭 ## 주의 첫번째에 돌리는 건 추천하지 않는다. 
# 데이터 훈련

results = model.train(
    data='/content/dataset/my_data/custom_data.yaml',
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE
)

# 시간 파일 생성

import pandas as pd
import os

# 1. 경로 설정
csv_path = f"/content/runs/detect/train{FOLDER_ID}/results.csv"

if os.path.exists(csv_path):
    # 2. CSV 로드 및 컬럼명 정리
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]  # 공백 제거

    # YOLO 버전에 따라 'val/time', 'train/time' 또는 그냥 'time'일 수 있습니다.
    # 시간 관련 컬럼을 자동으로 찾습니다.
    time_col = [c for c in df.columns if 'time' in c.lower()][0]

    # 3. 데이터 추출
    # 마지막 행 가져오기
    last_row = df.iloc[-1]

    final_epoch = int(last_row['epoch'])      # 최종 에포크 숫자
    total_time_seconds = last_row[time_col]   # 마지막 행의 누적 총 시간 (초)

    # 4. 평균 시간 계산 (이미지당이 아닌, 에포크당 평균 학습 시간)
    # 각 행의 시간 차이를 구하거나 전체 시간을 에포크 수로 나눕니다.
    avg_epoch_time = total_time_seconds / len(df)


    # 6. 리포트 생성
    csv_report = f"""[ YOLO CSV Train Report ]
Project Name       : {PROJECT_NAME}
---------------------------------------
Final Epoch        : {final_epoch}
Total Seconds      : {total_time_seconds:.2f} s
Avg Time per Epoch : {avg_epoch_time:.2f} s
---------------------------------------
Used Column        : {time_col}
"""

    # 7. 저장 및 드라이브 복사
    txt_filename = f"{PROJECT_NAME}_csv_time_analysis.txt"
    with open(txt_filename, "w") as f:
        f.write(csv_report)


# 결과 생성

weight_path = f'/content/runs/detect/train{FOLDER_ID}/weights/best.pt'
model = YOLO(weight_path)

results_predict = model.predict(source ='/content/dataset/my_data/test/images', save=True)
metrics = model.val(split='test')

# 결과 시간 생성

# [1. 검증 실행
# [2. 전체 공정 시간(AVG) 계산]
# speed 딕셔너리의 모든 값(전처리, 추론, 후처리)을 합산합니다.
avg_full_process_time = sum(metrics.speed.values())

# [3. 리포트 생성]
val_avg_report = f"""[ YOLO Validation Total Process Time ]
Project Name       : {PROJECT_NAME}
---------------------------------------
AVG Total Time     : {avg_full_process_time:.2f} ms / image

"""

# [4. 파일 저장 및 드라이브 복사]
val_txt_filename = f"{PROJECT_NAME}_val_total_avg.txt"
with open(val_txt_filename, "w") as f:
    f.write(val_avg_report)

# 파일 저장

import locale
locale.getpreferredencoding = lambda: "UTF-8"

import os

# 경로 자동 생성
num_str = str(FOLDER_ID)
TRAIN_DIR = f"/content/runs/detect/train{num_str}"
TEST_DIR = f"/content/runs/detect/val{num_str}"
PREDICT_DIR = f"/content/runs/detect/predict{num_str}"

# 번호에 따른 폴더 경로 자동 생성
num_str = str(FOLDER_ID)
TRAIN_DIR = f"/content/runs/detect/train{num_str}"
TEST_DIR = f"/content/runs/detect/val{num_str}"
PREDICT_DIR = f"/content/runs/detect/predict{num_str}"

# 2. 가중치 파일 복사
print(f"📦 가중치 파일 복사 중...")
!cp {TRAIN_DIR}/weights/best.pt {DRIVE_PATH}/{PROJECT_NAME}_weight.pt

# 3. Train/Test 리스트 정의 (쉼표 누락 수정 완료)
train_files = [
    ("confusion_matrix.png", "train_con.jpg"),
    ("BoxF1_curve.png", "train_f1.jpg"), # YOLO 버전에 따라 BoxF1 또는 F1일 수 있음
    ("BoxPR_curve.png", "train_pr.jpg"),
    ("results.png", "train_result.jpg")
]

test_files = [
    ("confusion_matrix.png", "test_con.jpg"),
    ("BoxF1_curve.png", "test_f1.jpg"),
    ("BoxPR_curve.png", "test_pr.jpg")
]

# 4. 파일 복사 실행
print(f"📊 그래프 파일 복사 중...")
for src, dst in train_files:
    if os.path.exists(f"{TRAIN_DIR}/{src}"):
        !cp {TRAIN_DIR}/{src} {DRIVE_PATH}/{PROJECT_NAME}_{dst}

for src, dst in test_files:
    if os.path.exists(f"{TEST_DIR}/{src}"):
        !cp {TEST_DIR}/{src} {DRIVE_PATH}/{PROJECT_NAME}_{dst}

# 5. Predict 결과 압축 및 전송
print(f"🖼️ Predict 결과 압축 및 드라이브 전송 중...")
ZIP_NAME = f"{PROJECT_NAME}_predict_results.zip"
!zip -r /content/{ZIP_NAME} {PREDICT_DIR}
!cp /content/{ZIP_NAME} {DRIVE_PATH}/{ZIP_NAME}

# 6. 시간 정리_train
!cp {txt_filename} {DRIVE_PATH}/{txt_filename}
# 시간 정리 test
!cp {val_txt_filename} {DRIVE_PATH}/{val_txt_filename}

print(f"\n✨ 모든 작업이 완료되었습니다!")
print(f"📍 저장 위치: {DRIVE_PATH}")
'''

In [15]:
# 데이터 훈련

results = model.train(
    data='/content/dataset/my_data/custom_data.yaml',
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/my_data/custom_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

In [30]:
import pandas as pd
import os

# 1. 경로 설정
csv_path = f"/content/runs/detect/train{FOLDER_ID}/results.csv"

if os.path.exists(csv_path):
    # 2. CSV 로드 및 컬럼명 정리
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]  # 공백 제거

    # YOLO 버전에 따라 'val/time', 'train/time' 또는 그냥 'time'일 수 있습니다.
    # 시간 관련 컬럼을 자동으로 찾습니다.
    time_col = [c for c in df.columns if 'time' in c.lower()][0]

    # 3. 데이터 추출
    # 마지막 행 가져오기
    last_row = df.iloc[-1]

    final_epoch = int(last_row['epoch'])      # 최종 에포크 숫자
    total_time_seconds = last_row[time_col]   # 마지막 행의 누적 총 시간 (초)

    # 4. 평균 시간 계산 (이미지당이 아닌, 에포크당 평균 학습 시간)
    # 각 행의 시간 차이를 구하거나 전체 시간을 에포크 수로 나눕니다.
    avg_epoch_time = total_time_seconds / len(df)


    # 6. 리포트 생성
    csv_report = f"""[ YOLO CSV Train Report ]
Project Name       : {PROJECT_NAME}
---------------------------------------
Final Epoch        : {final_epoch}
Total Seconds      : {total_time_seconds:.2f} s
Avg Time per Epoch : {avg_epoch_time:.2f} s
---------------------------------------
Used Column        : {time_col}
"""

    # 7. 저장 및 드라이브 복사
    txt_filename = f"{PROJECT_NAME}_csv_time_analysis.txt"
    with open(txt_filename, "w") as f:
        f.write(csv_report)

여기서 부터 결과

In [28]:
weight_path = f'/content/runs/detect/train{FOLDER_ID}/weights/best.pt'
model = YOLO(weight_path)

In [29]:
results_predict = model.predict(source ='/content/dataset/my_data/test/images', save=True)


image 1/10 /content/dataset/my_data/test/images/webcamimage_mix16.jpg: 768x1024 1 Car, 1 Dummy, 14.5ms
image 2/10 /content/dataset/my_data/test/images/webcamimage_mix20.jpg: 768x1024 1 Car, 1 Dummy, 6.8ms
image 3/10 /content/dataset/my_data/test/images/webcamimage_mix27.jpg: 768x1024 1 Car, 1 Dummy, 7.3ms
image 4/10 /content/dataset/my_data/test/images/webcamimage_mix35.jpg: 768x1024 1 Car, 1 Dummy, 7.0ms
image 5/10 /content/dataset/my_data/test/images/webcamimage_mix36.jpg: 768x1024 1 Car, 1 Dummy, 7.2ms
image 6/10 /content/dataset/my_data/test/images/webcamimage_mix43.jpg: 768x1024 1 Car, 1 Dummy, 7.4ms
image 7/10 /content/dataset/my_data/test/images/webcamimage_mix48.jpg: 768x1024 1 Car, 1 Dummy, 7.0ms
image 8/10 /content/dataset/my_data/test/images/webcamimage_mix8.jpg: 768x1024 1 Car, 1 Dummy, 6.9ms
image 9/10 /content/dataset/my_data/test/images/webcamimage_mix9.jpg: 768x1024 1 Car, 1 Dummy, 7.2ms
image 10/10 /content/dataset/my_data/test/images/webcamimage_mix94.jpg: 768x1024 1

In [31]:
metrics = model.val(split='test')

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2237.1±740.6 MB/s, size: 56.6 KB)
val: Scanning /content/dataset/my_data/test/labels.cache... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 3.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8it/s 0.4s
                   all         10         20      0.988          1      0.995       0.75
                   Car         10         10      0.995          1      0.995       0.71
                 Dummy         10         10      0.981          1      0.995       0.79
Speed: 1.8ms preprocess, 4.3ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val2


In [34]:
# [1. 검증 실행
# [2. 전체 공정 시간(AVG) 계산]
# speed 딕셔너리의 모든 값(전처리, 추론, 후처리)을 합산합니다.
avg_full_process_time = sum(metrics.speed.values())

# [3. 리포트 생성]
val_avg_report = f"""[ YOLO Validation Total Process Time ]
Project Name       : {PROJECT_NAME}
---------------------------------------
AVG Total Time     : {avg_full_process_time:.2f} ms / image

"""

# [4. 파일 저장 및 드라이브 복사]
val_txt_filename = f"{PROJECT_NAME}_val_total_avg.txt"
with open(val_txt_filename, "w") as f:
    f.write(val_avg_report)



print(f"✅ 전체 공정 평균 시간 측정 완료!")
print(val_avg_report)

✅ 전체 공정 평균 시간 측정 완료!
[ YOLO Validation Total Process Time ]
Project Name       : web_y8_b32_p25
---------------------------------------
AVG Total Time     : 7.72 ms / image




In [11]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [35]:
import os

# 경로 자동 생성
num_str = str(FOLDER_ID)
TRAIN_DIR = f"/content/runs/detect/train{num_str}"
TEST_DIR = f"/content/runs/detect/val{num_str}"
PREDICT_DIR = f"/content/runs/detect/predict{num_str}"

# 번호에 따른 폴더 경로 자동 생성
num_str = str(FOLDER_ID)
TRAIN_DIR = f"/content/runs/detect/train{num_str}"
TEST_DIR = f"/content/runs/detect/val{num_str}"
PREDICT_DIR = f"/content/runs/detect/predict{num_str}"

# 2. 가중치 파일 복사
print(f"📦 가중치 파일 복사 중...")
!cp {TRAIN_DIR}/weights/best.pt {DRIVE_PATH}/{PROJECT_NAME}_weight.pt

# 3. Train/Test 리스트 정의 (쉼표 누락 수정 완료)
train_files = [
    ("confusion_matrix.png", "train_con.jpg"),
    ("BoxF1_curve.png", "train_f1.jpg"), # YOLO 버전에 따라 BoxF1 또는 F1일 수 있음
    ("BoxPR_curve.png", "train_pr.jpg"),
    ("results.png", "train_result.jpg")
]

test_files = [
    ("confusion_matrix.png", "test_con.jpg"),
    ("BoxF1_curve.png", "test_f1.jpg"),
    ("BoxPR_curve.png", "test_pr.jpg")
]

# 4. 파일 복사 실행
print(f"📊 그래프 파일 복사 중...")
for src, dst in train_files:
    if os.path.exists(f"{TRAIN_DIR}/{src}"):
        !cp {TRAIN_DIR}/{src} {DRIVE_PATH}/{PROJECT_NAME}_{dst}

for src, dst in test_files:
    if os.path.exists(f"{TEST_DIR}/{src}"):
        !cp {TEST_DIR}/{src} {DRIVE_PATH}/{PROJECT_NAME}_{dst}

# 5. Predict 결과 압축 및 전송
print(f"🖼️ Predict 결과 압축 및 드라이브 전송 중...")
ZIP_NAME = f"{PROJECT_NAME}_predict_results.zip"
!zip -r /content/{ZIP_NAME} {PREDICT_DIR}
!cp /content/{ZIP_NAME} {DRIVE_PATH}/{ZIP_NAME}

# 6. 시간 정리_train
!cp {txt_filename} {DRIVE_PATH}/{txt_filename}
# 시간 정리 test
!cp {val_txt_filename} {DRIVE_PATH}/{val_txt_filename}

print(f"\n✨ 모든 작업이 완료되었습니다!")
print(f"📍 저장 위치: {DRIVE_PATH}")

📦 가중치 파일 복사 중...
📊 그래프 파일 복사 중...
🖼️ Predict 결과 압축 및 드라이브 전송 중...
  adding: content/runs/detect/predict2/ (stored 0%)
  adding: content/runs/detect/predict2/webcamimage_mix36.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix27.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix48.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix20.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix35.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix43.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix9.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix94.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix16.jpg (deflated 0%)
  adding: content/runs/detect/predict2/webcamimage_mix8.jpg (deflated 0%)

✨ 모든 작업이 완료되었습니다!
📍 저장 위치: /content/drive/MyDrive/rokey_face/result
